# Warm up a Qwen3 parent for OpenEnv’s REPL

This notebook runs an assistant-only LoRA update on conversations replayed in OpenEnv. The two adapted AlphaXiv conversations are a smoke dataset for testing the training path. To test whether a warmup helps on HotpotQA, use a larger set of verified demonstrations and reserve separate HotpotQA questions for evaluation.

Select **Runtime → Change runtime type → T4 GPU**, if available. Upload the `laguna-alphaxiv-replayed.jsonl` file generated by `replay_alphaxiv_parents.py`. The preflight runs on CPU and uses only tokenizer files.

In [ ]:
# This FP16 LoRA example does not use Colab's optional TorchAO integration.
%pip uninstall -y torchao
# Keep Arrow within the range supported by Colab's bundled cuDF.
%pip install -q "trl[peft] @ git+https://github.com/huggingface/trl.git@99bbdfec75ad1a57bb2df9c72278b391508eda5f" "transformers==5.23.0" "datasets==5.0.0" "pyarrow>=21,<24"

## Load and inspect the replayed conversations

Run the setup cell in a fresh session, before importing any training libraries. If you already ran a previous version, restart the session after setup, then resume here. The notebook checks that all four model code turns in the smoke examples receive loss and that the longest conversation fits in the configured sequence length.

In [ ]:
from pathlib import Path

from datasets import load_dataset
from google.colab import files
from transformers import AutoTokenizer

from trl.chat_template_utils import get_training_chat_template


MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
MAX_LENGTH = 6144
SMOKE_RUN = True
DATA_PATH = Path("/content/laguna-alphaxiv-replayed.jsonl")

if not DATA_PATH.is_file():
    uploaded = files.upload()
    if DATA_PATH.name not in uploaded:
        raise ValueError(f"Upload a JSONL file named {DATA_PATH.name}.")
    DATA_PATH.write_bytes(uploaded[DATA_PATH.name])

dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
if not dataset:
    raise ValueError("The replay file contains no conversations.")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
template = get_training_chat_template(tokenizer, require_prefix_preservation=False)

In [ ]:
from itertools import groupby


rows = []
for index, record in enumerate(dataset):
    messages = record["messages"]
    tokens = tokenizer.apply_chat_template(
        messages,
        chat_template=template,
        tokenize=True,
        return_dict=True,
        return_assistant_tokens_mask=True,
    )
    ids = tokens["input_ids"]
    mask = tokens["assistant_masks"]
    assistant_turns = [message["content"] for message in messages if message["role"] == "assistant"]
    masked_spans = [
        tokenizer.decode([token_id for token_id, _ in run])
        for selected, run in groupby(zip(ids, mask, strict=True), key=lambda pair: pair[1])
        if selected
    ]
    if len(ids) > MAX_LENGTH:
        raise ValueError(f"Row {index} has {len(ids)} tokens and would be truncated at {MAX_LENGTH}.")
    if (
        not assistant_turns
        or len(masked_spans) != len(assistant_turns)
        or any(
            span.strip().removeprefix("<think>\n\n</think>").removesuffix("<|im_end|>").strip() != turn.strip()
            for span, turn in zip(masked_spans, assistant_turns, strict=True)
        )
    ):
        raise ValueError(f"Row {index} does not mask exactly the parent assistant turns.")
    rows.append((len(ids), sum(mask)))
    if index < 5:
        print(f"Row {index}: {len(ids)} total tokens, {sum(mask)} assistant tokens")

print(f"{len(rows)} conversations; longest is {max(length for length, _ in rows)} tokens")

## Run one optimizer step and save the adapter

The default smoke run does one update. For a larger verified warmup dataset, set `SMOKE_RUN = False` to run one epoch. This uses a small model, FP16, LoRA, and gradient checkpointing for a single T4. Take a baseline and compare on held-out HotpotQA before drawing conclusions about learning.

In [ ]:
import math

import torch
from peft import LoraConfig

from trl import SFTConfig, SFTTrainer


if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running this cell.")
print(torch.cuda.get_device_name(0))
torch.cuda.reset_peak_memory_stats()

config = SFTConfig(
    output_dir="/content/rlm-warmup",
    model_init_kwargs={"revision": MODEL_REVISION, "dtype": torch.float16, "attn_implementation": "sdpa"},
    max_length=MAX_LENGTH,
    assistant_only_loss=True,
    packing=False,
    per_device_train_batch_size=1,
    activation_checkpointing=True,
    fp16=True,
    bf16=False,
    learning_rate=1e-4,
    num_train_epochs=1,
    max_steps=1 if SMOKE_RUN else -1,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)
trainer = SFTTrainer(
    model=MODEL_ID,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    ),
)
result = trainer.train()
if not math.isfinite(result.training_loss) or (SMOKE_RUN and result.global_step != 1):
    raise RuntimeError(f"Unexpected training result: {result}")
print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
trainer.save_model("/content/rlm-warmup/adapter")
tokenizer.save_pretrained("/content/rlm-warmup/adapter")

In [ ]:
import shutil


archive = shutil.make_archive("/content/rlm-warmup-adapter", "zip", "/content/rlm-warmup/adapter")
files.download(archive)